# Cell features

`extract_cell_features` runs the whole pipeline for a cell (spikes, sweep, sweepset
and cell features), like Allen's `extract_cell_features` with
`cell_extractor_for_nwb`. Its `"cell"` entry holds the cell features under the
names of the Allen Cell Types Database (`ri`, `tau`, `vrest`, `f_i_curve_slope`,
`threshold_v_long_square`, ...). The examples use a cell from the [Allen Cell Types Database](https://celltypes.brain-map.org), loaded with a helper of this repository's tests. With your own recordings, all you need are NumPy arrays of time `t` (s), membrane potential `v` (mV) and, optionally, the injected current `i` (pA).

In [1]:
import sys
from pathlib import Path

import jax
import numpy as np
import pandas as pd

jax.config.update("jax_enable_x64", True)  # float64: results match Allen exactly

# Recordings from the Allen Cell Types Database, as downloaded for the tests
# (uv run python tests/load_testdata.py), read with a helper of the test suite.
REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(REPO))

from jaxley_ephys import extract_cell_features  # noqa: E402
from jaxley_ephys.cell import LONG_SQUARE_WINDOW, RAMP_START  # noqa: E402

In [2]:
from tests.utils.allen_dataset import load_specimen_case  # noqa: E402 (uses AllenSDK)

case = load_specimen_case(519832676, data_root=str(REPO / "tests/testdata/allen_cache"))

Each stimulus type is a list of sweeps `(t, v, i, start, end)`; sweeps may
differ in length. Allen's analysis windows: long squares from 1.02 to 2.02 s,
ramps from 1.02 s and short squares from the start of the recording epoch, until
the end of the sweep (`end=None`).

In [3]:
def sweeps(kind, window=None):
    out = []
    for record in case.records(kind=kind):
        t, v, i, start, end = case.trace_for(record.sweep_number)
        out.append((t, v, i, *(window or (start, None))))
    return out


cell = extract_cell_features(
    sweeps("long_square", LONG_SQUARE_WINDOW),
    sweeps("short_square"),
    sweeps("ramp", (RAMP_START, None)),
)
print(list(cell))
print(
    f"rheobase {float(cell['long_squares']['rheobase_i']):.0f} pA, "
    f"input resistance {float(cell['cell']['ri']):.1f} MOhm"
)

['long_squares', 'short_squares', 'ramps', 'cell']
rheobase 130 pA, input resistance 146.9 MOhm


## Comparison with the Allen Cell Types Database

The database lists these features for the cell. They agree; the database follows
a few extra conventions of its own (see `allen_ephys/ALLEN_EPHYS_FIXES.md`).

In [4]:
def db_value(name):
    try:
        return float(case.feature_row[name])
    except (KeyError, ValueError):
        return np.nan


pd.DataFrame(
    {
        "jaxley_ephys": {name: float(x) for name, x in cell["cell"].items()},
        "allen_db": {name: db_value(name) for name in cell["cell"]},
    }
)

,jaxley_ephys,allen_db
adaptation,0.179194,0.179194
avg_isi,245.906667,245.906667
latency,0.086620,0.086620
f_i_curve_slope,0.057143,0.057143
ri,146.874792,146.874636
sag,0.160742,0.160742
tau,50.052259,50.049031
vm_for_sag,-84.218755,-84.218758
vrest,-72.547669,-72.547661
fast_trough_t_long_square,1.198760,1.198760
